# 🔥 PySpark Complete Guide — Google Colab (Local Mode)

Welcome to this step-by-step PySpark tutorial designed for **Google Colab** in **local mode**.

---

## 📋 What You'll Learn

| # | Topic |
|---|-------|
| 1 | PySpark Development Environment |
| 2 | Core PySpark Operations |
| 3 | Understanding Spark Execution |
| 4 | Data Sources and Formats |
| 5 | PySpark SQL |
| 6 | Advanced DataFrame Operations |
| 7 | Basic Performance Concepts |
| 8 | User-Defined Functions (UDFs) |
| 9 | Data Quality and Validation |
| 10 | Introduction to Structured Streaming |
| 11 | Advanced Debugging |
| 12 | PySpark Integration with Public APIs |

---

> **📌 How to use this notebook:**  
> Run each cell **top to bottom** using `Shift + Enter`. Each section is self-contained after the setup in Section 1.

---
# 1. 🛠️ PySpark Development Environment

## What is PySpark?

**Apache Spark** is a distributed data processing engine designed for speed and ease of use. **PySpark** is the Python API for Spark, allowing you to write Spark applications in Python.

## What is a SparkSession?

The `SparkSession` is the **single entry point** to all Spark functionality. Think of it as your connection to the Spark engine. Everything you do in PySpark starts with a `SparkSession`.

```
Your Python Code
      ↓
  SparkSession
      ↓
  Spark Engine (Local Mode on this machine)
      ↓
  Processes Data
```

## What is Local Mode?

Spark can run in two modes:

| Mode | Description | Use Case |
|------|-------------|----------|
| **Local Mode** | Runs on your single machine using threads | Development, learning, small datasets |
| **Cluster Mode** | Runs across many machines | Production, large-scale data |

In Google Colab, we use **local mode** — `local[*]` means Spark uses **all available CPU cores** on the Colab VM.

## Why Google Colab?

Google Colab provides a **free hosted Python environment** with GPU/CPU resources. We can install PySpark directly using `pip` and run it in local mode without any cluster setup.

In [ ]:
# ── Step 1: Install PySpark ──────────────────────────────────────────────────
# This installs PySpark from PyPI. The quiet flag (-q) suppresses verbose output.
!pip install pyspark -q

print("✅ PySpark installed successfully!")

In [ ]:
# ── Step 2: Create a SparkSession ────────────────────────────────────────────
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySpark_Tutorial")          # Name your application
    .master("local[*]")                   # local[*] = use all CPU cores
    .config("spark.ui.showConsoleProgress", "false")  # Cleaner output
    .getOrCreate()                        # Get existing session or create new
)

# Verify the session is working
print(f"✅ SparkSession created")
print(f"   Spark Version : {spark.version}")
print(f"   App Name      : {spark.sparkContext.appName}")
print(f"   Master        : {spark.sparkContext.master}")

In [ ]:
# ── Step 3: Create Sample Data Files ─────────────────────────────────────────
# We'll generate small sample files (CSV, JSON, Parquet) to use throughout
# this notebook. In a real scenario, these would be pre-existing data files.

import json
import os

os.makedirs("/tmp/spark_data", exist_ok=True)

# ── CSV Sample ────────────────────────────────────────────────────────────────
csv_content = """id,name,department,salary,age
1,Alice,Engineering,95000,30
2,Bob,Marketing,72000,35
3,Charlie,Engineering,88000,28
4,Diana,HR,65000,40
5,Eve,Engineering,102000,33
6,Frank,Marketing,78000,29
7,Grace,HR,68000,45
8,Hank,Engineering,91000,31
9,Ivy,Marketing,75000,27
10,Jack,HR,70000,38
"""
with open("/tmp/spark_data/employees.csv", "w") as f:
    f.write(csv_content)

# ── JSON Sample ───────────────────────────────────────────────────────────────
# Spark reads JSON in "JSON Lines" format (one JSON object per line)
json_records = [
    {"product_id": 101, "product": "Laptop",  "category": "Electronics", "price": 999.99, "stock": 50},
    {"product_id": 102, "product": "Mouse",   "category": "Electronics", "price": 29.99,  "stock": 200},
    {"product_id": 103, "product": "Desk",    "category": "Furniture",   "price": 349.99, "stock": 30},
    {"product_id": 104, "product": "Chair",   "category": "Furniture",   "price": 249.99, "stock": 45},
    {"product_id": 105, "product": "Notebook","category": "Stationery",  "price": 4.99,   "stock": 500},
]
with open("/tmp/spark_data/products.json", "w") as f:
    for record in json_records:
        f.write(json.dumps(record) + "\n")

# ── Parquet Sample ────────────────────────────────────────────────────────────
# We'll create the Parquet file using Spark itself (most reliable method)
orders_data = [
    (1001, 1, 101, 2, "2024-01-15"),
    (1002, 2, 102, 3, "2024-01-16"),
    (1003, 1, 103, 1, "2024-01-17"),
    (1004, 3, 101, 1, "2024-01-18"),
    (1005, 4, 105, 10,"2024-01-19"),
]
orders_df = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "product_id", "quantity", "order_date"]
)
orders_df.write.mode("overwrite").parquet("/tmp/spark_data/orders.parquet")

print("✅ Sample data files created:")
print("   /tmp/spark_data/employees.csv")
print("   /tmp/spark_data/products.json")
print("   /tmp/spark_data/orders.parquet")

In [ ]:
# ── Step 4: Basic DataFrame Creation ─────────────────────────────────────────
# A DataFrame is Spark's primary data structure — similar to a pandas DataFrame
# but distributed across machines in a cluster.

# Method 1: From a Python list of tuples
data = [("Alice", 30, "Engineer"),
        ("Bob",   25, "Analyst"),
        ("Carol", 35, "Manager")]

df = spark.createDataFrame(data, columns=["name", "age", "role"])

# Method 2: Read from CSV file we created
emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")

# ── Display the DataFrames ────────────────────────────────────────────────────
print("=" * 40)
print("DataFrame from Python list:")
print("=" * 40)
df.show()                    # show() is an ACTION — triggers computation
df.printSchema()             # Shows column names and data types

print("=" * 40)
print("DataFrame from CSV (first 5 rows):")
print("=" * 40)
emp_df.show(5)

print(f"\nNumber of rows in employees dataset: {emp_df.count()}")
print(f"Number of columns: {len(emp_df.columns)}")
print(f"Column names: {emp_df.columns}")

---
# 2. ⚙️ Core PySpark Operations

## Transformations vs Actions

PySpark operations fall into two categories:

| Type | What It Does | Examples | Returns |
|------|-------------|---------|--------|
| **Transformation** | Defines a new DataFrame from an existing one | `select`, `filter`, `groupBy`, `join` | A new DataFrame (lazy) |
| **Action** | Triggers actual computation and returns results | `show`, `count`, `collect`, `write` | A value or side effect |

## Key Transformations

- **`select()`** — Choose specific columns (like SQL `SELECT`)
- **`filter()` / `where()`** — Keep rows matching a condition (like SQL `WHERE`)
- **`withColumn()`** — Add or replace a column
- **`groupBy()`** — Group rows by column values (like SQL `GROUP BY`)
- **`join()`** — Combine two DataFrames on a common key

## Key Actions

- **`show(n)`** — Print first `n` rows
- **`count()`** — Return number of rows
- **`collect()`** — Return all rows to the driver as a Python list

> **⚠️ Important:** Transformations are **lazy** — they don't run until an action is called. This allows Spark to optimize the full computation plan before executing.

In [ ]:
# ── select, filter, withColumn ───────────────────────────────────────────────
from pyspark.sql import functions as F

# Load employees data (we created this in Section 1)
emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")

# ── select: choose specific columns ──────────────────────────────────────────
print("[select] Name and Department only:")
emp_df.select("name", "department").show()

# ── filter: keep rows matching a condition ────────────────────────────────────
print("[filter] Engineering department only:")
emp_df.filter(emp_df.department == "Engineering").show()

# You can also use SQL-style string expressions
print("[filter] Salary > 80000:")
emp_df.filter("salary > 80000").select("name", "salary").show()

# ── withColumn: add a new computed column ────────────────────────────────────
print("[withColumn] Add monthly_salary column:")
emp_df.withColumn("monthly_salary", F.round(F.col("salary") / 12, 2)).select("name", "salary", "monthly_salary").show()

In [ ]:
# ── groupBy and Aggregations ─────────────────────────────────────────────────
# groupBy() groups rows. Aggregation functions then summarize each group.
# Common aggregation functions: count(), sum(), avg(), min(), max()

# Average salary by department
print("Average salary per department:")
emp_df.groupBy("department") \
      .agg(
          F.count("id").alias("headcount"),
          F.avg("salary").alias("avg_salary"),
          F.max("salary").alias("max_salary"),
          F.min("salary").alias("min_salary")
      ) \
      .orderBy("avg_salary", ascending=False) \
      .show()

# Count employees per department
print("Employee count by department:")
emp_df.groupBy("department").count().show()

In [ ]:
# ── Joins ─────────────────────────────────────────────────────────────────────
# Joins combine two DataFrames based on a matching column (key).
# Spark supports: inner, left, right, full outer, semi, anti joins.

# Create two small DataFrames to demonstrate joins
employees = spark.createDataFrame([
    (1, "Alice", 10),
    (2, "Bob",   20),
    (3, "Carol", 10),
    (4, "Dave",  30),   # dept 30 doesn't exist in departments
], ["emp_id", "name", "dept_id"])

departments = spark.createDataFrame([
    (10, "Engineering"),
    (20, "Marketing"),
    (40, "Finance"),    # dept 40 has no employees
], ["dept_id", "dept_name"])

# INNER JOIN — only matching rows from both DataFrames
print("[INNER JOIN] Employees with their department:")
employees.join(departments, on="dept_id", how="inner").show()

# LEFT JOIN — all rows from left + matched rows from right (NULLs if no match)
print("[LEFT JOIN] All employees, with or without a department:")
employees.join(departments, on="dept_id", how="left").show()

# FULL OUTER JOIN — all rows from both (NULLs where no match)
print("[FULL OUTER JOIN] All employees AND all departments:")
employees.join(departments, on="dept_id", how="full").show()

---
# 3. 🔍 Understanding Spark Execution

## Lazy Evaluation — The Core Concept

Spark uses **lazy evaluation**: when you write transformations (like `filter`, `select`, `groupBy`), Spark does **not** execute them immediately. It builds a **logical plan** (a recipe). Execution only happens when you call an **action** (like `show()`, `count()`, `collect()`).

```
Step 1: df.filter(...)       → No execution! Spark remembers the plan.
Step 2: .select(...)         → No execution! Plan grows.
Step 3: .groupBy(...)        → No execution! Plan grows.
Step 4: .show()              → ACTION! Now Spark executes the full plan.
```

## Why Lazy Evaluation?

Lazy evaluation allows Spark's **Catalyst Optimizer** to:
- Reorder operations for efficiency
- Push filters down closer to the data source (avoid reading unnecessary data)
- Combine multiple operations into a single pass over the data

## Actions vs Transformations — Quick Reference

| Transformations (Lazy) | Actions (Eager) |
|------------------------|------------------|
| `select()` | `show()` |
| `filter()` | `count()` |
| `groupBy()` | `collect()` |
| `withColumn()` | `take(n)` |
| `join()` | `write()` |
| `orderBy()` | `first()` |

In [ ]:
# ── Demonstrating Lazy Evaluation ────────────────────────────────────────────
import time

emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")

# Building the transformation chain (NO COMPUTATION YET)
print("Building transformation chain...")
t0 = time.time()

transformed = (emp_df
               .filter("salary > 70000")                        # Transformation 1
               .withColumn("bonus", F.col("salary") * 0.10)     # Transformation 2
               .select("name", "department", "salary", "bonus") # Transformation 3
               .orderBy("salary", ascending=False))              # Transformation 4

t1 = time.time()
print(f"   Time to build plan (NO computation): {(t1-t0)*1000:.1f} ms")
print("   ↳ Spark has only created a plan — no data was processed yet!\n")

# Now trigger an ACTION — this is when computation actually happens
print("Triggering ACTION: show()")
t2 = time.time()
transformed.show()          # ← Actual execution happens HERE
t3 = time.time()
print(f"   Time to execute (with computation): {(t3-t2)*1000:.1f} ms")
print("   ↳ Now Spark ran all 4 transformations together in one optimized job!")

In [ ]:
# ── Different Types of Actions ───────────────────────────────────────────────
emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")

# ACTION 1: count() — returns a single integer
row_count = emp_df.count()
print(f"count()   → {row_count} rows")

# ACTION 2: first() — returns a single Row object
first_row = emp_df.first()
print(f"first()   → {first_row}")

# ACTION 3: take(n) — returns a Python list of Row objects
top3 = emp_df.take(3)
print(f"take(3)   → {top3}")

# ACTION 4: collect() — returns ALL rows as a Python list
# ⚠️ WARNING: Never use collect() on large DataFrames — it loads ALL data into
# driver memory and can cause OutOfMemoryError. Fine for small datasets.
all_rows = emp_df.select("name").collect()
names = [row["name"] for row in all_rows]
print(f"collect() → {names}")

# ACTION 5: show() — prints rows to console (doesn't return Python object)
print("\nshow(3):")
emp_df.show(3)

---
# 4. 📂 Data Sources and Formats

## Common File Formats in Spark

| Format | Pros | Cons | Best For |
|--------|------|------|----------|
| **CSV** | Human-readable, universal | No schema, slow, large files | Simple data exchange |
| **JSON** | Flexible, supports nesting | Verbose, slow | Semi-structured data |
| **Parquet** | Columnar, compressed, fast | Not human-readable | Analytics, production |
| **ORC** | Highly optimized for Hive | Less common | Hive/Hadoop ecosystems |

## Schema Inference vs Explicit Schema

When reading data, Spark can:
1. **Infer the schema automatically** (`inferSchema=True`) — convenient but requires reading the file twice and may guess wrong types
2. **Use an explicit schema you define** — faster, safer, and gives you full control over data types

## Data Types in PySpark

PySpark's `types` module provides: `StringType`, `IntegerType`, `LongType`, `DoubleType`, `FloatType`, `BooleanType`, `DateType`, `TimestampType`, `ArrayType`, `StructType` (for nested/structured data)

In [ ]:
# ── Read and Write CSV, JSON, Parquet ────────────────────────────────────────

# ── READ CSV ──────────────────────────────────────────────────────────────────
csv_df = (
    spark.read
    .option("header", True)        # First row is column names
    .option("inferSchema", True)   # Automatically detect data types
    .csv("/tmp/spark_data/employees.csv")
)
print("CSV Schema:")
csv_df.printSchema()

# ── READ JSON ─────────────────────────────────────────────────────────────────
json_df = spark.read.json("/tmp/spark_data/products.json")
print("JSON Schema:")
json_df.printSchema()
json_df.show()

# ── READ PARQUET ──────────────────────────────────────────────────────────────
parquet_df = spark.read.parquet("/tmp/spark_data/orders.parquet")
print("Parquet Schema:")
parquet_df.printSchema()
parquet_df.show()

# ── WRITE in different formats ────────────────────────────────────────────────
# Write the employees CSV as Parquet (better format for analytics)
csv_df.write.mode("overwrite").parquet("/tmp/spark_data/employees_output.parquet")
print("✅ Written: employees_output.parquet")

# Write as JSON
csv_df.write.mode("overwrite").json("/tmp/spark_data/employees_output.json")
print("✅ Written: employees_output.json")

# Write as CSV with header
csv_df.write.mode("overwrite").option("header", True).csv("/tmp/spark_data/employees_output_copy.csv")
print("✅ Written: employees_output_copy.csv")

In [ ]:
# ── Define and Apply a Custom Schema ─────────────────────────────────────────
# Instead of letting Spark guess types, we define them explicitly.
# This is the recommended approach in production.

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define the schema manually
employee_schema = StructType([
    StructField("id",         IntegerType(), nullable=False),  # nullable=False means required
    StructField("name",       StringType(),  nullable=True),
    StructField("department", StringType(),  nullable=True),
    StructField("salary",     DoubleType(),  nullable=True),
    StructField("age",        IntegerType(), nullable=True),
])

# Read CSV using the custom schema (no inference needed)
typed_df = (
    spark.read
    .option("header", True)
    .schema(employee_schema)   # Apply our schema instead of inferring
    .csv("/tmp/spark_data/employees.csv")
)

print("Schema with explicit type definitions:")
typed_df.printSchema()

print("Data with enforced types:")
typed_df.show(5)

# Confirm salary is DoubleType (not just string)
print(f"Salary column type: {dict(typed_df.dtypes)['salary']}")

---
# 5. 🗄️ PySpark SQL

## What is PySpark SQL?

PySpark allows you to run **standard SQL queries** directly on your DataFrames. This is useful if you:
- Already know SQL and want to leverage that knowledge
- Need to share code with people who know SQL but not Python
- Want to express complex queries more naturally

## Temporary Views

To query a DataFrame with SQL, you first register it as a **temporary view**. A temporary view is like a virtual table — it exists only for the duration of your SparkSession and doesn't write any data to disk.

```python
df.createOrReplaceTempView("my_table")    # Register as temp view
result = spark.sql("SELECT * FROM my_table WHERE age > 30")
```

## SQL vs DataFrame API

Both approaches are **functionally equivalent** — Spark converts them to the same execution plan:

```python
# These two produce identical results:
df.filter("age > 30").select("name")             # DataFrame API
spark.sql("SELECT name FROM t WHERE age > 30")   # SQL API
```

You can freely mix both styles in the same application.

In [ ]:
# ── Create Temporary Views ────────────────────────────────────────────────────
emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")
products_df = spark.read.json("/tmp/spark_data/products.json")
orders_df = spark.read.parquet("/tmp/spark_data/orders.parquet")

# Register DataFrames as temporary SQL views
emp_df.createOrReplaceTempView("employees")
products_df.createOrReplaceTempView("products")
orders_df.createOrReplaceTempView("orders")

# List all available views in the current catalog
print("Registered temporary views:")
spark.catalog.listTables()
for table in spark.catalog.listTables():
    print(f"   - {table.name} ({table.tableType})")

In [ ]:
# ── Run SQL Queries on Views ──────────────────────────────────────────────────

# Simple SELECT with WHERE
print("[SQL] Engineering employees earning > 90k:")
spark.sql("""
    SELECT name, salary
    FROM employees
    WHERE department = 'Engineering'
      AND salary > 90000
    ORDER BY salary DESC
""").show()

# GROUP BY with aggregates
print("[SQL] Department salary stats:")
spark.sql("""
    SELECT
        department,
        COUNT(*)        AS headcount,
        ROUND(AVG(salary), 2) AS avg_salary,
        MAX(salary)     AS top_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""").show()

# JOIN across views
print("[SQL] Order details with product names:")
spark.sql("""
    SELECT
        o.order_id,
        o.order_date,
        p.product,
        o.quantity,
        ROUND(p.price * o.quantity, 2) AS total_value
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    ORDER BY total_value DESC
""").show()

In [ ]:
# ── Mixing SQL and DataFrame API ──────────────────────────────────────────────
# SQL result is just a regular DataFrame — you can chain DataFrame operations on it.

# Step 1: Use SQL to get a result set
high_earners_sql = spark.sql("""
    SELECT name, department, salary
    FROM employees
    WHERE salary > 80000
""")

# Step 2: Chain DataFrame API on top of the SQL result
result = (
    high_earners_sql
    .withColumn("tax_bracket", F.lit("High"))      # Add a literal column
    .withColumn("net_salary",  F.col("salary") * 0.75)   # 25% tax deduction
    .orderBy("net_salary", ascending=False)
)

print("SQL result + DataFrame transformations combined:")
result.show()

---
# 6. 🔧 Advanced DataFrame Operations

## Conditional Transformations

PySpark's `when()` / `otherwise()` functions are the equivalent of SQL's `CASE WHEN` — they let you create new column values based on conditions:

```python
F.when(condition, value_if_true)
 .when(another_condition, another_value)
 .otherwise(default_value)
```

## Nested Data Structures

Real-world data often contains **arrays** and **structs** (nested objects). PySpark provides functions to work with these:

- **`explode()`** — Expands an array column so each element becomes its own row
- **`col("struct.field")`** — Access nested struct fields using dot notation
- **`array()`** — Create array columns
- **`struct()`** — Create struct (nested object) columns
- **`size()`** — Get the length of an array

In [ ]:
# ── when / otherwise — Conditional Column Logic ──────────────────────────────
from pyspark.sql.functions import when

emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")

# Add salary_band column based on salary ranges
result = emp_df.withColumn(
    "salary_band",
    when(F.col("salary") >= 100000, "Platinum")
    .when(F.col("salary") >= 85000,  "Gold")
    .when(F.col("salary") >= 70000,  "Silver")
    .otherwise("Bronze")            # Default case
)

print("Salary bands based on conditional logic:")
result.select("name", "salary", "salary_band").show()

# Use when() inside aggregations
print("Count by salary band:")
result.groupBy("salary_band").count().orderBy("salary_band").show()

In [ ]:
# ── explode() and Nested Structures ──────────────────────────────────────────
from pyspark.sql.functions import explode, col, struct, array

# Create a DataFrame with an array column
team_data = [
    ("Project Alpha", "Engineering", ["Alice", "Bob", "Carol"]),
    ("Project Beta",  "Marketing",   ["Dave", "Eve"]),
    ("Project Gamma", "HR",          ["Frank"]),
]
teams_df = spark.createDataFrame(team_data, ["project", "department", "members"])

print("Original DataFrame with array column:")
teams_df.show(truncate=False)

# explode() turns each array element into a separate row
print("After explode() — one row per member:")
teams_df.withColumn("member", explode("members")) \
        .select("project", "department", "member") \
        .show()

# ── Nested struct columns ─────────────────────────────────────────────────────
# Create a DataFrame with a struct (nested) column
people_df = spark.createDataFrame([
    ("Alice", {"city": "Hyderabad", "state": "Telangana"}),
    ("Bob",   {"city": "Bengaluru", "state": "Karnataka"}),
], ["name", "address"])

print("DataFrame with nested struct column:")
people_df.printSchema()
people_df.show(truncate=False)

# Access nested fields using dot notation
print("Extracting nested struct fields:")
people_df.select(
    col("name"),
    col("address.city").alias("city"),
    col("address.state").alias("state")
).show()

---
# 7. ⚡ Basic Performance Concepts (Demo Level)

## Partitions — The Unit of Parallelism

Spark divides data into **partitions** — smaller chunks that can be processed in **parallel** by different tasks. More partitions = more parallelism (up to the number of available cores).

```
DataFrame (1 million rows)
├── Partition 1 (250k rows) → Task 1 → Core 1
├── Partition 2 (250k rows) → Task 2 → Core 2
├── Partition 3 (250k rows) → Task 3 → Core 3
└── Partition 4 (250k rows) → Task 4 → Core 4
```

## repartition() vs coalesce()

| Method | Description | Shuffle? | Use When |
|--------|-------------|----------|----------|
| `repartition(n)` | Redistribute data into exactly `n` partitions | Yes (full shuffle) | Increasing partitions, or when data is skewed |
| `coalesce(n)` | Merge partitions down to `n` | No (or minimal) | Reducing partitions efficiently before writing |

> **Rule of thumb:** Use `coalesce()` to reduce and `repartition()` to increase or balance.

## Broadcast Join

In a normal join, Spark shuffles both DataFrames across the network. A **broadcast join** instead sends the **smaller** DataFrame to every node — eliminating the expensive shuffle.

Use broadcast joins when one DataFrame is **small enough to fit in memory** (typically < 10 MB).

In [ ]:
# ── repartition vs coalesce Demo ─────────────────────────────────────────────
emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")

# Check current number of partitions
print(f"Default partitions after CSV read : {emp_df.rdd.getNumPartitions()}")

# repartition() — full shuffle, increases OR decreases partitions
repartitioned = emp_df.repartition(4)
print(f"After repartition(4)              : {repartitioned.rdd.getNumPartitions()}")

# coalesce() — efficient merge, only decreases partitions (no full shuffle)
coalesced = repartitioned.coalesce(2)
print(f"After coalesce(2)                 : {coalesced.rdd.getNumPartitions()}")

# Partition by a column — useful before writing partitioned data to disk
emp_df.repartition(3, "department").write.mode("overwrite").parquet("/tmp/spark_data/emp_partitioned")
print("\n✅ Written partitioned output (3 partitions, by department)")

# Coalesce to 1 partition (single file output) — common for small result sets
emp_df.coalesce(1).write.mode("overwrite").option("header", True).csv("/tmp/spark_data/emp_single_file")
print("✅ Written single-file CSV output")

In [ ]:
# ── Broadcast Join Example ────────────────────────────────────────────────────
from pyspark.sql.functions import broadcast

# Large table (in production this could be millions of rows)
large_df = emp_df  # employees — the "large" side

# Small lookup table (small enough to fit in memory of each executor)
dept_codes = spark.createDataFrame([
    ("Engineering", "ENG"),
    ("Marketing",   "MKT"),
    ("HR",          "HRS"),
], ["department", "dept_code"])

# Normal join (Spark might shuffle both sides)
normal_result = large_df.join(dept_codes, on="department", how="left")
print("Normal join result:")
normal_result.select("name", "department", "dept_code").show()

# Broadcast join — explicitly tells Spark to broadcast the small table
# Spark sends dept_codes to every worker instead of shuffling employees
broadcast_result = large_df.join(broadcast(dept_codes), on="department", how="left")
print("Broadcast join result (same output, faster execution on large data):")
broadcast_result.select("name", "department", "dept_code").show()

print("✅ Both produce identical results — broadcast just changes HOW Spark executes the join")

---
# 8. 🐍 User-Defined Functions (UDFs)

## What is a UDF?

A **User-Defined Function (UDF)** lets you apply a custom Python function to each row of a Spark DataFrame column. UDFs bridge the gap between Spark's built-in functions and custom business logic.

## When to Use UDFs

Use a UDF when you **can't** express your logic using Spark's built-in functions (`pyspark.sql.functions`):

| Prefer Built-in Functions | Use UDF |
|--------------------------|----------|
| String operations (`upper`, `trim`, `split`) | Complex custom string parsing |
| Math (`round`, `abs`, `pow`) | Custom math models |
| Date operations (`year`, `month`, `datediff`) | Non-standard date formats |

## ⚠️ Performance Note

Python UDFs have overhead because data must be **serialized** between the JVM (where Spark runs) and the Python interpreter for every row. Always prefer **built-in Spark functions** when possible. If you need UDFs for performance-critical code, consider **Pandas UDFs** (vectorized UDFs) instead.

In [ ]:
# ── Create and Use a Python UDF ───────────────────────────────────────────────
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType

emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")

# ── UDF 1: Classify salary into a descriptive tier ────────────────────────────
def salary_tier(salary):
    """Custom function to assign salary tier labels."""
    if salary is None:
        return "Unknown"
    elif salary >= 100000:
        return "Senior"
    elif salary >= 80000:
        return "Mid-Senior"
    elif salary >= 65000:
        return "Mid-Level"
    else:
        return "Junior"

# Register the Python function as a Spark UDF
# Must specify the return type — Spark needs this for schema planning
salary_tier_udf = udf(salary_tier, StringType())

# Apply the UDF as a new column
result = emp_df.withColumn("tier", salary_tier_udf(F.col("salary")))
print("UDF — salary tier classification:")
result.select("name", "salary", "tier").show()

# ── UDF 2: Generate initials from name ────────────────────────────────────────
def get_initials(name):
    """Return first letter of each word in the name."""
    if name is None:
        return ""
    return ".".join(word[0].upper() for word in name.split()) + "."

initials_udf = udf(get_initials, StringType())

print("UDF — generate initials from name:")
emp_df.withColumn("initials", initials_udf(F.col("name"))) \
      .select("name", "initials") \
      .show()

# ── Register UDF for use in SQL queries too ───────────────────────────────────
spark.udf.register("salary_tier_sql", salary_tier, StringType())
emp_df.createOrReplaceTempView("employees")

print("UDF used inside SQL query:")
spark.sql("""
    SELECT name, salary, salary_tier_sql(salary) AS tier
    FROM employees
    ORDER BY salary DESC
""").show()

In [ ]:
def upper(name: str) -> str:
    ...

# Vectorized UDFs (Pandas UDFs) in PySpark

## What are Vectorized UDFs?

Regular Python UDFs process data **one row at a time** — Spark serializes each row to Python, applies the function, and sends it back. This is slow due to constant Python ↔ JVM context switching.

**Vectorized UDFs** (also called **Pandas UDFs**) process data in **columnar batches** using Apache Arrow for zero-copy data transfer. Instead of receiving a single value, your function receives a **Pandas Series or DataFrame**, letting you use vectorized NumPy/Pandas operations.

```
Regular UDF:     row1 → python → row1 result
                 row2 → python → row2 result   ← slow, row-by-row
                 row3 → python → row3 result

Pandas UDF:      [row1, row2, row3, ...] → python (batch) → [result1, result2, result3]
                                                              ← fast, columnar batch
```

---

## Key Differences

| Feature | Regular UDF | Pandas (Vectorized) UDF |
|---|---|---|
| Input type | Single Python value | `pd.Series` / `pd.DataFrame` |
| Processing | Row-by-row | Batch (Arrow columnar) |
| Speed | Slow | 10–100x faster |
| Null handling | Manual | Pandas native |
| Decorator | `@udf` | `@pandas_udf` |

---

## Types of Pandas UDFs

### 1. Series → Series (most common)
Takes one or more Series, returns a Series. Equivalent to your row-wise UDFs.

```python
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType, IntegerType, DoubleType
import pandas as pd

# ── Vectorized UDF 1: Salary Tier (replaces your salary_tier_udf) ─────────────
@pandas_udf(StringType())
def salary_tier_vectorized(salary: pd.Series) -> pd.Series:
    """
    Receives the entire 'salary' column as a Pandas Series (a batch).
    Uses pd.cut for fast vectorized binning — no Python for-loop needed.
    """
    return pd.cut(
        salary,
        bins=[-float("inf"), 65000, 80000, 100000, float("inf")],
        labels=["Junior", "Mid-Level", "Mid-Senior", "Senior"],
        right=False                # left-inclusive: [65000, 80000)
    ).astype(str).where(salary.notna(), "Unknown")  # handle nulls cleanly

result = emp_df.withColumn("tier", salary_tier_vectorized(F.col("salary")))
result.select("name", "salary", "tier").show()
```

```python
# ── Vectorized UDF 2: Get Initials (replaces your initials_udf) ───────────────
@pandas_udf(StringType())
def get_initials_vectorized(name: pd.Series) -> pd.Series:
    """
    .str accessor gives vectorized string ops across the whole column at once.
    No Python loop — Pandas handles each element internally in C.
    """
    return name.str.split()                          # split each name into words
               .apply(                               # apply only for the join logic
                   lambda words: ".".join(w[0].upper() for w in words) + "."
                   if isinstance(words, list) else ""
               )

emp_df.withColumn("initials", get_initials_vectorized(F.col("name"))) \
      .select("name", "initials") \
      .show()
```

---

### 2. Series → Scalar (aggregate UDF)
Returns a **single scalar** from a whole column — used inside `groupBy().agg()`.

```python
from pyspark.sql.functions import pandas_udf, PandasUDFType

# ── Aggregate Pandas UDF: Custom trimmed mean (drops top & bottom 10%) ────────
@pandas_udf(DoubleType())
def trimmed_mean(salary: pd.Series) -> float:
    """
    Called once per group. Returns one float per group.
    Perfect for custom aggregations not available in Spark built-ins.
    """
    low, high = salary.quantile(0.10), salary.quantile(0.90)
    trimmed = salary[(salary >= low) & (salary <= high)]
    return trimmed.mean()

emp_df.groupBy("department") \
      .agg(trimmed_mean(F.col("salary")).alias("trimmed_avg_salary")) \
      .show()
```

---

### 3. Iterator of Series → Iterator of Series (amortized setup cost)
Best when your UDF needs to **load a model or resource once** and reuse it across batches.

```python
from typing import Iterator

# ── Iterator UDF: Load a model/resource ONCE, apply across all batches ─────────
@pandas_udf(StringType())
def classify_with_model(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:
    """
    The model is loaded ONCE per executor (not once per row or batch).
    Ideal for ML inference, regex compilation, DB connection setup, etc.
    """
    # ✅ Expensive setup happens here — outside the loop
    import re
    senior_pattern = re.compile(r"engineer|manager|lead", re.IGNORECASE)

    for batch in iterator:                    # batch is a pd.Series
        yield batch.apply(
            lambda title: "Senior Track" if senior_pattern.search(str(title)) else "Other"
        )

emp_df.withColumn("track", classify_with_model(F.col("job_title"))).show()
```

---

### 4. Multiple Columns → Series
Pass multiple columns using `F.struct()` or as separate arguments.

```python
# ── Multi-column Pandas UDF: Compute compensation ratio ──────────────────────
@pandas_udf(DoubleType())
def compensation_ratio(salary: pd.Series, bonus: pd.Series) -> pd.Series:
    """
    Receives two columns as two separate Series — fully vectorized math.
    """
    total = salary + bonus.fillna(0)
    return (total / salary).round(3)          # pure NumPy arithmetic, very fast

emp_df.withColumn(
    "comp_ratio",
    compensation_ratio(F.col("salary"), F.col("bonus"))
).show()
```

---

## SQL Registration (same as regular UDFs)

```python
# Pandas UDFs can also be registered for SQL use
spark.udf.register("salary_tier_sql", salary_tier_vectorized)

spark.sql("""
    SELECT name, salary, salary_tier_sql(salary) AS tier
    FROM employees
    ORDER BY salary DESC
""").show()
```

---

## When to Use Which

```
Need row-wise transformation?          → @pandas_udf (Series → Series)
Need custom aggregation?               → @pandas_udf (Series → Scalar) in groupBy
Loading ML model / heavy resource?    → Iterator of Series → Iterator of Series
Simple built-in math/string?          → Use Spark built-ins (F.when, F.regexp_extract)
Legacy code / simple one-off?         → Regular @udf is fine, but slower
```

The golden rule: **prefer Spark built-ins > Pandas UDFs > regular UDFs** for performance.

---
# 9. ✅ Data Quality and Validation

## Why Data Quality Matters

Real-world data is often **dirty**: missing values, wrong types, duplicate records, outliers. Before running any analysis, you should validate and clean your data. Common issues:

- **NULL values** — missing data that can cause incorrect aggregations
- **Duplicates** — double-counting records
- **Type mismatches** — text in a numeric column
- **Outliers** — extreme values that skew statistics
- **Invalid values** — negative ages, future birthdays, etc.

## Key PySpark Functions for Data Quality

| Function | Purpose |
|----------|---------|
| `isNull()` / `isNotNull()` | Check for NULLs |
| `na.drop()` | Drop rows with NULLs |
| `na.fill()` | Replace NULLs with a default value |
| `dropDuplicates()` | Remove duplicate rows |
| `describe()` | Basic stats (count, mean, std, min, max) |
| `summary()` | Extended statistics |
| `countDistinct()` | Count unique values |

In [ ]:
# ── Null Handling and Filtering Bad Records ───────────────────────────────────

# Create a dataset with intentional quality issues
messy_data = [
    (1, "Alice",   "Engineering",  95000.0, 30),
    (2, "Bob",     None,           72000.0, 35),    # NULL department
    (3, None,      "Engineering",  88000.0, 28),    # NULL name
    (4, "Diana",   "HR",          -65000.0, 40),    # Negative salary (invalid!)
    (5, "Eve",     "Engineering",  None,    33),    # NULL salary
    (6, "Frank",   "Marketing",   78000.0, 200),   # Unrealistic age
    (7, "Grace",   "HR",          68000.0,  45),
    (7, "Grace",   "HR",          68000.0,  45),   # Duplicate row!
]

messy_df = spark.createDataFrame(messy_data, ["id", "name", "department", "salary", "age"])

print("Original messy data:")
messy_df.show()

# ── Step 1: Identify nulls ────────────────────────────────────────────────────
print("NULL counts per column:")
null_counts = messy_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in messy_df.columns
])
null_counts.show()

# ── Step 2: Drop rows where critical columns are null ────────────────────────
no_null_names = messy_df.na.drop(subset=["name"])  # Drop rows with NULL name
print(f"Rows after dropping NULL names: {no_null_names.count()} (was {messy_df.count()})")

# ── Step 3: Fill NULLs with defaults ─────────────────────────────────────────
filled_df = messy_df.na.fill({
    "department": "Unknown",
    "salary": 0.0
})
print("After filling NULLs with defaults:")
filled_df.show()

# ── Step 4: Filter out invalid values ────────────────────────────────────────
valid_df = (
    messy_df
    .filter(F.col("name").isNotNull())       # Must have a name
    .filter(F.col("salary") > 0)             # Salary must be positive
    .filter(F.col("age").between(18, 100))   # Age must be realistic
    .dropDuplicates()                        # Remove exact duplicates
)
print(f"Clean records after all validation: {valid_df.count()}")
valid_df.show()

In [ ]:
# ── Basic Data Profiling ──────────────────────────────────────────────────────
# Profiling helps you understand your data before transforming it.

emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")

# Basic statistics: count, mean, stddev, min, max for numeric columns
print("📊 Basic Statistics (describe):")
emp_df.describe(["salary", "age"]).show()

# Extended statistics: adds percentiles (25%, 50%, 75%)
print("📊 Extended Statistics (summary):")
emp_df.select("salary", "age").summary().show()

# Distinct values per column
print("📊 Distinct value counts:")
for col_name in ["department"]:
    distinct_count = emp_df.select(col_name).distinct().count()
    print(f"   {col_name}: {distinct_count} distinct values")

# Value distribution for categorical columns
print("\n📊 Department distribution:")
emp_df.groupBy("department") \
      .agg(F.count("*").alias("count"),
           F.round(F.count("*") / emp_df.count() * 100, 1).alias("pct%")) \
      .orderBy("count", ascending=False) \
      .show()

---
# 10. 🌊 Introduction to Structured Streaming (Basic)

## What is Structured Streaming?

**Structured Streaming** is Spark's framework for processing **real-time data streams**. Instead of running a batch job on a static dataset, Structured Streaming processes data incrementally as it arrives.

Think of it as: **Spark treats a data stream as an unbounded (never-ending) table** that keeps growing, and continuously queries it.

```
Data Source (Kafka, files, sockets, etc.)
      ↓ (rows keep arriving)
Streaming DataFrame  ← you define transformations
      ↓
Output Sink (console, memory, files, Kafka)
      ↓
Results update continuously as new data arrives
```

## Key Concepts

| Concept | Description |
|---------|-------------|
| **Source** | Where data comes from (Kafka, files, rate source for testing) |
| **Sink** | Where results go (console, memory, files, Kafka) |
| **Trigger** | How often to process new data (every 2 seconds, once, continuous) |
| **Output Mode** | `append` (only new rows), `complete` (full table), `update` (changed rows) |

## Rate Source

For learning and testing, Spark provides a built-in **rate source** that generates rows at a specified rate (rows per second). Perfect for experimenting without a real data source like Kafka.

In [ ]:
# ── Structured Streaming with Rate Source ─────────────────────────────────────
import time

# Create a streaming DataFrame from the built-in "rate" source
# rate source generates rows: (timestamp, value) where value increments from 0
streaming_df = (
    spark.readStream
    .format("rate")                     # Built-in rate generator
    .option("rowsPerSecond", 5)          # Generate 5 rows per second
    .load()
)

print("Streaming DataFrame schema:")
streaming_df.printSchema()
print(f"Is streaming: {streaming_df.isStreaming}")

# Add a transformation — classify even vs odd values
transformed_stream = streaming_df.withColumn(
    "parity",
    when(F.col("value") % 2 == 0, "even").otherwise("odd")
)

In [ ]:
# ── Write Stream to Memory Sink ───────────────────────────────────────────────
# Memory sink stores the streaming output in an in-memory table.
# Useful for testing — lets you query it with SQL.

query = (
    transformed_stream
    .writeStream
    .format("memory")               # Store results in memory (for testing)
    .queryName("rate_table")         # Name the in-memory table
    .outputMode("append")            # Append new rows as they arrive
    .trigger(processingTime="2 seconds")  # Process every 2 seconds
    .start()
)

print(f"✅ Streaming query started. Status: {query.status['message']}")

# Wait a few seconds for data to accumulate
time.sleep(6)

# Query the in-memory table using SQL
print("\nData captured in memory sink (last 10 rows):")
spark.sql("SELECT * FROM rate_table ORDER BY timestamp DESC LIMIT 10").show(truncate=False)

# Check how many rows have been processed
row_count = spark.sql("SELECT COUNT(*) as total FROM rate_table").collect()[0][0]
print(f"Total rows processed so far: {row_count}")

# Stop the query
query.stop()
print("\n✅ Streaming query stopped.")

In [ ]:
# ── Stream to Console Sink with Trigger Once ──────────────────────────────────
# Trigger.Once() processes all available data exactly once and then stops.
# Useful for scheduled micro-batch jobs.

from pyspark.sql.streaming import Trigger

console_query = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
    .withColumn("category",
        when(F.col("value") % 3 == 0, "fizz")
        .when(F.col("value") % 5 == 0, "buzz")
        .otherwise("other")
    )
    .writeStream
    .format("console")              # Print to console
    .outputMode("append")
    .option("numRows", 10)          # Show max 10 rows per batch
    .option("truncate", False)
    .trigger(once=True)             # Process one batch and stop
    .start()
)

console_query.awaitTermination(timeout=15)  # Wait up to 15 seconds
print("✅ One-time trigger query complete.")

---
# 11. 🐛 Advanced Debugging (Basic)

## Why Debugging in Spark is Unique

Debugging PySpark is more complex than debugging regular Python because:

1. **Lazy evaluation** — Errors don't surface until an action is triggered
2. **Distributed execution** — The actual error may occur on a worker, not the driver
3. **Long stack traces** — Spark's JVM stack traces can be very verbose
4. **Data-dependent errors** — A bad value in row #1,000,000 only fails when that partition runs

## Practical Debugging Strategies

| Strategy | Description |
|----------|-------------|
| **`printSchema()`** | Verify column names and types before transformations |
| **`show()`** | Inspect a sample of data at each stage |
| **`explain()`** | View Spark's physical execution plan |
| **`try/except`** | Catch and handle errors gracefully |
| **Python `logging`** | Structured logging instead of raw `print()` |
| **`count()` checkpoints** | Verify row counts at stages to spot data loss |

In [ ]:
# ── try/except with Spark Operations ────────────────────────────────────────
from pyspark.sql.utils import AnalysisException

# ── Example 1: Catch a AnalysisException (bad column name) ───────────────────
emp_df = spark.read.option("header", True).option("inferSchema", True).csv("/tmp/spark_data/employees.csv")

try:
    # Intentional error: referencing a column that doesn't exist
    bad_result = emp_df.select("name", "nonexistent_column")
    bad_result.show()   # Error surfaces at the ACTION
except AnalysisException as e:
    print(f"⚠️  AnalysisException caught (schema problem):")
    print(f"   {str(e)[:120]}...")
    print(f"   ↳ Available columns: {emp_df.columns}")

# ── Example 2: Catch a general exception from bad file path ──────────────────
try:
    missing_df = spark.read.parquet("/tmp/spark_data/this_file_does_not_exist.parquet")
    missing_df.count()   # Action triggers the error
except Exception as e:
    print(f"\n⚠️  File not found error caught:")
    print(f"   {type(e).__name__}: {str(e)[:100]}...")

# ── Example 3: Defensive coding pattern ──────────────────────────────────────
def safe_read_csv(path, spark_session):
    """Read a CSV file with error handling. Returns None on failure."""
    try:
        df = spark_session.read.option("header", True).option("inferSchema", True).csv(path)
        row_count = df.count()  # Validate by triggering an action
        print(f"✅ Successfully loaded '{path}' with {row_count} rows")
        return df
    except Exception as e:
        print(f"❌ Failed to load '{path}': {type(e).__name__}")
        return None

print("\nDefensive read function:")
df1 = safe_read_csv("/tmp/spark_data/employees.csv", spark)
df2 = safe_read_csv("/tmp/spark_data/missing.csv", spark)

In [ ]:
# ── Logging in PySpark Applications ──────────────────────────────────────────
# Use Python's logging module instead of print() for structured, level-aware output.

import logging
import sys

# ── Configure the logger ──────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout
)
logger = logging.getLogger("PySpark_Pipeline")

# ── Simulate a data pipeline with logging ────────────────────────────────────
def run_pipeline(input_path):
    logger.info(f"Pipeline starting — reading from: {input_path}")

    try:
        # Step 1: Read
        df = spark.read.option("header", True).option("inferSchema", True).csv(input_path)
        logger.info(f"[Step 1] Data loaded — {df.count()} rows, {len(df.columns)} columns")

        # Step 2: Validate
        null_count = df.filter(F.col("salary").isNull()).count()
        if null_count > 0:
            logger.warning(f"[Step 2] Found {null_count} rows with NULL salary — these will be dropped")
        else:
            logger.info("[Step 2] Validation passed — no NULL salaries")

        # Step 3: Transform
        clean_df = df.filter(F.col("salary").isNotNull())
        result_df = clean_df.withColumn("annual_bonus", F.col("salary") * 0.10)
        logger.info(f"[Step 3] Transformation complete — {result_df.count()} valid records")

        # Step 4: Output
        result_df.write.mode("overwrite").parquet("/tmp/spark_data/pipeline_output")
        logger.info("[Step 4] Results written to parquet successfully")
        logger.info("✅ Pipeline completed successfully")
        return result_df

    except Exception as e:
        logger.error(f"❌ Pipeline FAILED at step with error: {str(e)[:100]}")
        raise

# Run the pipeline
output = run_pipeline("/tmp/spark_data/employees.csv")

---
# 12. 🌐 PySpark Integration — Fetching from a Public API

## API-Based Data Ingestion

One common pattern in data pipelines is **fetching data from external APIs** and then processing it with Spark. The typical flow is:

```
External REST API
     ↓ (HTTP request using Python's requests library)
Python List / Dict
     ↓ (spark.createDataFrame())
Spark DataFrame
     ↓ (transformations + actions)
Processed Output
```

## How It Works

1. Use Python's `requests` library to call the API and get JSON data
2. Parse the JSON into a Python list of dictionaries
3. Convert to a Spark DataFrame using `spark.createDataFrame()` or by reading from JSON
4. Apply all the Spark transformations you've learned

> **Note:** In this example, we use the free public API `https://jsonplaceholder.typicode.com` — a popular fake REST API for testing and prototyping.

In [ ]:
# ── Fetch Data from a Public API ──────────────────────────────────────────────
import requests

# Fetch posts from JSONPlaceholder — a free test REST API
print("Fetching data from JSONPlaceholder API...")
response = requests.get("https://jsonplaceholder.typicode.com/posts")

# Check the response status
print(f"HTTP Status Code : {response.status_code}")
if response.status_code == 200:
    posts_json = response.json()    # Parse JSON response into Python list
    print(f"Records fetched  : {len(posts_json)}")
    print(f"Sample record    : {posts_json[0]}")
else:
    print("API call failed — using static fallback data")
    posts_json = [
        {"userId": 1, "id": 1, "title": "Sample post 1", "body": "Content 1"},
        {"userId": 1, "id": 2, "title": "Sample post 2", "body": "Content 2"},
        {"userId": 2, "id": 3, "title": "Sample post 3", "body": "Content 3"},
    ]

In [ ]:
# ── Convert API Response to Spark DataFrame ───────────────────────────────────

# Method: Convert Python list of dicts → Spark DataFrame
# Spark can infer schema from Python dicts automatically
posts_df = spark.createDataFrame(posts_json)

print("Schema of API data:")
posts_df.printSchema()

print(f"Total posts: {posts_df.count()}")
print("\nSample data:")
posts_df.select("id", "userId", "title").show(5, truncate=50)

# ── Now apply Spark transformations ──────────────────────────────────────────

# Add derived columns
enriched_df = posts_df.withColumn(
    "title_length", F.length(F.col("title"))
).withColumn(
    "body_word_count", F.size(F.split(F.col("body"), " "))
)

# Aggregate posts per user
print("Post activity per user (from API data):")
enriched_df.groupBy("userId") \
           .agg(
               F.count("id").alias("post_count"),
               F.round(F.avg("title_length"), 1).alias("avg_title_len"),
               F.round(F.avg("body_word_count"), 1).alias("avg_body_words")
           ) \
           .orderBy("userId") \
           .show(10)

# Save the result
enriched_df.write.mode("overwrite").parquet("/tmp/spark_data/api_posts_output")
print("\n✅ API data processed and saved as Parquet")

---
# 🎉 Summary — What You've Learned

Congratulations! You've completed the PySpark Complete Guide. Here's a recap of everything covered:

| Section | Key Takeaway |
|---------|-------------|
| **1. Environment Setup** | SparkSession is the entry point; Colab runs PySpark in local mode |
| **2. Core Operations** | `select`, `filter`, `withColumn`, `groupBy`, joins — the daily toolkit |
| **3. Lazy Evaluation** | Transformations build a plan; actions trigger execution |
| **4. Data Formats** | Parquet is fastest for analytics; always prefer explicit schemas |
| **5. PySpark SQL** | Register views and run SQL — fully interchangeable with DataFrame API |
| **6. Advanced Ops** | `when/otherwise` for conditionals; `explode` for nested arrays |
| **7. Performance** | `repartition` to balance; `coalesce` to merge; broadcast small tables |
| **8. UDFs** | Custom Python logic in Spark — use sparingly, prefer built-ins |
| **9. Data Quality** | Always validate: handle NULLs, remove duplicates, filter invalid rows |
| **10. Streaming** | Rate source for testing; memory/console sinks for learning |
| **11. Debugging** | Use `try/except`, structured logging, and `explain()` to diagnose issues |
| **12. API Integration** | Fetch → Python list → `createDataFrame()` → Transform |

---

## 🚀 Next Steps

1. **Practice** — Run each section with your own datasets
2. **MLlib** — Explore Spark's machine learning library
3. **Delta Lake** — Learn about ACID transactions on Spark
4. **Pandas UDFs** — Vectorized UDFs for better performance
5. **Databricks** — A managed Spark platform widely used in industry
6. **Real Streaming Sources** — Connect to Kafka, EventHub, or Kinesis

---

> **💡 Pro Tip:** The best way to master PySpark is to bring your own dataset and try to answer real questions about it using the techniques from this notebook!

In [ ]:
# ── Clean Up: Stop the SparkSession ──────────────────────────────────────────
# Always stop the SparkSession when you're done to free up resources.

spark.stop()
print("✅ SparkSession stopped. Resources released.")
print("🎓 Notebook complete — great work!")